(User_MolecularSystem_Get)=
# Getting attributes

This page documents `view.get(...)`.

You will use `get` whenever you want to bring information from the molecular system into Python: names and indices, counts, coordinates, box vectors, times, and so on.

`view.get(...)` is a small wrapper around MolSysMT’s `get`, so the mental model is the same.

If you want a deeper dive (more element-level examples and edge cases), see the MolSysMT tutorial: [Get](https://www.uibcdf.org/molsysmt/content/user/tools/basic/get).

## Common attribute names

Both selection strings and `get` rely on a shared vocabulary of attribute names (for atoms, groups, molecules, and so on).

You do not need to memorize these, but it helps to know what names exist so you can search and experiment confidently.

Here is a quick reference table with the most common attribute names you will see in MolSysMT/MolSysViewer:

```{include} ../_snippets/attributes_table.md
```

In [1]:
import molsysviewer as viewer

view = viewer.demo["181L"]
view.show()

In [2]:
from molsysviewer.thirds.jupyter import load_html_in_notebook

load_html_in_notebook("../../../_static/views/demo_181L.html")

## A first example

Let’s start with something simple: a system-level snapshot.

`get` works with **attribute flags**: you request values by passing `attribute_name=True` as keyword arguments.

When you are exploring a new system, this “ask a few questions quickly” pattern is usually the fastest way to orient yourself.

In [3]:
view.get(element="system", n_atoms=True, n_structures=True)

[1441, 1]

If you prefer a stable `{name: value}` mapping (often easier to read and debug), use `output_type="dictionary"`:

In [4]:
view.get(element="system", n_molecules=True, n_bonds=True, output_type="dictionary")

{'n_molecules': 141, 'n_bonds': 1322}

## Getting attributes at different element levels

One of the most useful ideas in MolSysMT/MolSysViewer is that you can ask questions at different *element* levels (see the Elements table in {doc}`topology`).

Two element levels that are especially useful early on are:

- `entity`: the molecular nature/type (for example, water vs benzene)
- `molecule`: a single molecular instance (for example, one water molecule)

Here is a quick way to see how many molecules belong to each entity:


In [5]:
view.get(element="entity", entity_name=True, n_molecules=True, output_type="dictionary")

{'entity_name': ['T4 LYSOZYME',
  'CHLORIDE ION',
  '2-HYDROXYETHYL DISULFIDE',
  'BENZENE',
  'water'],
 'n_molecules': [1, 2, 1, 1, 136]}

You can also combine `element` with a selection at that same level. For example, select the water entity and retrieve its molecule count:


In [6]:
view.get(element="entity", selection='entity_name=="water"', n_molecules=True)

[136]

## Getting multiple attributes

You can request multiple attributes in a single call. This is a great way to inspect a few elements in one step.

When you request several attributes at once, using `output_type="dictionary"` usually makes the output easier to read.

In [7]:
view.get(
    element="atom",
    selection=[0, 1, 2],
    atom_index=True,
    atom_name=True,
    group_name=True,
    chain_id=True,
    output_type="dictionary",
)

{'atom_index': [0, 1, 2],
 'atom_name': ['N', 'CA', 'C'],
 'group_name': ['MET', 'MET', 'MET'],
 'chain_id': ['A', 'A', 'A']}

For more examples (and more element-level coverage), the MolSysMT tutorial linked above is a good next stop.

## `selection` and `mask`

A common workflow is: select a subset, then retrieve attributes for it.

Both `selection` and `mask` can be given as indices or as selection strings (parsed by MolSysMT).

When both are present, `mask` acts as an additional intersection filter: it refines the elements picked by `selection`.

In [8]:
protein_ca = view.select(selection='molecule_type=="protein"', mask='atom_name=="CA"')
view.get(element="atom", selection=protein_ca[:5], atom_name=True, group_name=True, chain_id=True)

[['CA', 'CA', 'CA', 'CA', 'CA'],
 ['MET', 'ASN', 'ILE', 'PHE', 'GLU'],
 ['A', 'A', 'A', 'A', 'A']]

For example, start from a selection and then refine it with an additional mask:

In [9]:
view.get(element="atom", selection=protein_ca[:5], mask='group_name=="PHE"', atom_name=True, group_name=True, chain_id=True)

[['CA'], ['PHE'], ['A']]

If you want a deeper explanation of selection strings, see {doc}`selection`.

## Structural attributes and `structure_indices`

Structural attributes (like `coordinates`, `box`, or `time`) require `structure_indices`, because you need to specify which structure(s) you mean:

In [10]:
view.get(
    element="atom",
    selection=[0, 1, 2],
    structure_indices=0,
    coordinates=True,
)

Magnitude,[[[4.398199999999999 -0.3258 0.9163] [4.343399999999999 -0.19169999999999998 0.9134] [4.2006 -0.19659999999999997 0.964]]]
Units,nanometer


## A note about bonds

Some input formats do not include explicit bond connectivity. When you request bond-related attributes, MolSysMT can sometimes infer missing bonds on the fly.

You can control this behavior with the `get_missing_bonds` argument.

When bonds are available (or inferred), you can query simple connectivity questions. For example: “which atoms are bonded to these atoms?”

In [11]:
view.get(element="atom", selection=[0, 1, 2], bonded_atoms=True)

[[np.int64(1)],
 [np.int64(0), np.int64(2), np.int64(4)],
 [np.int64(1), np.int64(3), np.int64(8)]]

If you want the bond list explicitly (as atom pairs), you can request the bonded atom pairs found in a selection of atoms:

In [12]:
view.get(element="atom", selection=[0, 1, 2, 3, 4], bonded_atom_pairs=True)

[[0, 1], [1, 2], [1, 4], [2, 3], [2, 8], [4, 5]]

Sometimes you only want bonds fully contained in your selection. In that case, request the bonded atom pairs within the list itself (pairs where both atoms are in the selection):

In [13]:
view.get(element="atom", selection=[0, 1, 2, 3, 4], inner_bonded_atom_pairs=True)

[[0, 1], [1, 2], [1, 4], [2, 3]]